# 永続的なコンテキストのためのS3ベクトルメモリ

[Amazon S3 Vectors](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors.html)でセマンティックメモリを構築します。このノートブックでは、本番環境対応のインフラストラクチャで、すべてのセッションにわたって情報を記憶するエージェントを作成する方法を説明します。

## 学習内容

- Amazon S3 Vectorsでセッション間メモリを実装する
- エージェント知識のセマンティック検索を設定する
- 本番環境対応の永続メモリシステムを構築する
- セッションマネージャーとベクトルメモリアプローチを比較する

## 前提条件

- [ノートブック04: 状態とセッション](04-state-and-sessions.ipynb)を完了していること
- S3 Vectorsアクセス権を持つAWSアカウント
- ベクトルデータベースとセマンティック検索の理解

📚 **詳細**: [Amazon S3 Vectors はじめにガイド](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors-getting-started.html)

## セットアップ

`s3_vector_memory`ツールは必要なインフラストラクチャを自動的に作成します：
1. ✅ S3 Vectorバケット（存在しない場合は自動的に作成）
2. ✅ ベクトルインデックス（存在しない場合は自動的に作成）
3. ⚠️ 適切なAWS権限があることを確認してください：
   - `s3vectors:CreateVectorBucket`
   - `s3vectors:DescribeVectorBucket`
   - `s3vectors:CreateIndex`
   - `s3vectors:DescribeIndex`
   - `s3vectors:PutVectors`
   - `s3vectors:QueryVectors`
   - `bedrock:InvokeModel`（埋め込み用）

参照: [S3 Vectors はじめにガイド](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors-getting-started.html)

In [ ]:
!pip install strands-agents strands-agents-tools boto3 -q

In [ ]:
import boto3
import os
from strands import Agent
from strands.models import BedrockModel
from strands_tools import image_reader, file_read, use_llm
from video_reader_local import video_reader_local
from s3_memory import s3_vector_memory

print("✅ All imports successful!")

## 設定

S3 Vectors環境を設定します。

**注意**: ベクトルバケットとインデックスは、存在しない場合、初回使用時に自動的に作成されます！

In [ ]:
# S3 Vectors設定
# 重要: 最初にAWS_REGIONを設定してください - すべてのリソースはこのリージョンに作成されます
os.environ['AWS_REGION'] = 'us-east-1'                                    # AWSリージョン（バケット、インデックス、Bedrockがこれを使用します）

# 重要: 'your-unique-bucket-name'を一意のバケット名に置き換えてください
# S3バケット名はすべてのAWSアカウント間でグローバルに一意である必要があります
# 例: 'my-company-vector-store-2024' または 'username-strands-vectors'
os.environ['VECTOR_BUCKET_NAME'] = 'your-unique-bucket-name'              # ⚠️ これを変更してください！
os.environ['VECTOR_INDEX_NAME'] = 'strands-multimodal'                    # インデックス名（そのままでも可）
os.environ['EMBEDDING_MODEL'] = 'amazon.nova-2-multimodal-embeddings-v1:0'  # Amazon Nova Multimodal Embeddings

print(f"🌍 リージョンの設定: {os.environ['AWS_REGION']}")
print(f"📦 ベクトルバケット: {os.environ['VECTOR_BUCKET_NAME']}")
print(f"📊 ベクトルインデックス: {os.environ['VECTOR_INDEX_NAME']}")
print(f"🤖 埋め込みモデル: {os.environ['EMBEDDING_MODEL']}")

if os.environ['VECTOR_BUCKET_NAME'] == 'your-unique-bucket-name':
    print("\n⚠️  警告: VECTOR_BUCKET_NAMEを一意の名前に変更してください！")


In [ ]:
# メモリ分離のためのユーザー識別
USER_ID = "demo_user_reinvent"

print(f"👤 ユーザーID: {USER_ID}")


## S3ベクトルメモリツール

`s3_vector_memory`ツールは3つの操作を提供します：

1. **store**: ベクトルメモリに情報を保存
2. **retrieve**: 関連するメモリを検索
3. **list**: ユーザーのすべてのメモリを一覧表示

### 自動インフラストラクチャセットアップ

初回使用時、ツールは自動的に：
- S3 Vectorバケットが存在しない場合は作成
- 最適な設定（1024次元、COSINE距離）でベクトルインデックスを作成
- インデックスが準備できるまで待機

つまり、手動セットアップなしで、すぐにベクトルメモリの使用を開始できます！

In [ ]:
# メモリの保存をテスト
result = s3_vector_memory(
    action="store",
    content="ユーザーはバックエンド開発にPython、フロントエンドにReactを好みます。",
    user_id=USER_ID
)
print("保存結果:", result)

In [ ]:
# メモリの取得をテスト
result = s3_vector_memory(
    action="retrieve",
    query="プログラミングの好み",
    user_id=USER_ID
)
print("取得したメモリ:", result)

In [ ]:
result

## ベクトルメモリ付きエージェントの作成

永続的なベクトルメモリを持つマルチモーダルエージェントを作成しましょう：

In [ ]:
# Bedrockモデルのセットアップ
aws_region = os.environ.get('AWS_REGION', 'us-east-1')
session = boto3.Session(region_name=aws_region)
print(f"🌍 使用中のAWSリージョン: {aws_region}")
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    boto_session=session
)

# メモリ強化エージェント用のシステムプロンプト
MEMORY_SYSTEM_PROMPT = """あなたはセッションをまたいだ永続的なメモリを持つAIアシスタントです。

あなたの能力：
- **マルチモーダル分析**: 画像、ドキュメント、動画、テキストを処理
- **永続メモリ**: ユーザーの好み、洞察、コンテキストを記憶
- **セッション間理解**: 以前の会話からのメモリにアクセス
- **セマンティック検索**: 過去の対話から関連情報を見つける

メモリ使用ガイドライン：
1. 応答する前に、s3_vector_memoryを使用して関連するメモリを取得
2. 重要な洞察、好み、発見を保存
3. 関連する場合は以前の会話を参照
4. 時間をかけて理解を構築

コンテンツを処理する際：
1. コンテキストのために関連するメモリを取得
2. 新しいコンテンツを分析
3. 重要な洞察を保存
4. 新しい分析とメモリの両方を使用して包括的な応答を提供
"""

# ベクトルメモリ付きエージェントの作成
memory_agent = Agent(
    model=bedrock_model,
    tools=[
        s3_vector_memory,  # ベクトルメモリツール
        image_reader,      # 画像処理
        file_read,         # ドキュメント処理
        video_reader_local,      # 動画処理
        use_llm           # 高度な推論
    ],
    system_prompt=MEMORY_SYSTEM_PROMPT
)

print("✅ メモリ強化エージェントを作成しました！")
print("🧠 メモリバックエンド: Amazon S3 Vectors")

## 最初の対話: 好みの確立

In [ ]:
# 最初の会話 - ユーザーの好みを確立
response = memory_agent(
    f"""こんにちは！私はAWSの開発者アドボケートで、AIとサーバーレステクノロジーに取り組んでいます。
    特に興味があるのは：
    - マルチモーダルAI処理
    - サーバーレスアーキテクチャ
    - AWS Bedrockと生成AI
    - 本番環境対応のAIシステムの構築
    
    今後の会話のために、私の興味に関するこの情報を保存してください。
    
    ユーザーID: {USER_ID}"""
)

print(response)

## メモリコンテキストでの画像分析

In [ ]:
# メモリコンテキストでアーキテクチャ図を分析
response = memory_agent(
    f"""data-sample/diagram.jpgのアーキテクチャ図を分析してください。
    
    分析する前に：
    1. 私の興味と好みについてメモリを確認
    2. そのコンテキストを使用してパーソナライズされた分析を提供
    
    分析後：
    1. 重要なアーキテクチャの洞察を保存
    2. 私の興味に合う技術を記録
    
    私のユーザーID: {USER_ID}
    
    提供してください：
    - アーキテクチャの概要
    - 技術スタック
    - 観察されたベストプラクティス
    - 私の興味に基づく推奨事項"""
)

print(response)

## メモリ付きドキュメント処理

In [ ]:
# メモリ統合でドキュメントを処理
response = memory_agent(
    f"""ドキュメントdata-sample/Welcome-Strands-Agents-SDK.pdfを処理してください。
    
    メモリ強化処理：
    1. メモリから私の興味を取得
    2. そのコンテキストでドキュメントを処理
    3. 私の興味に関連する重要な洞察を保存
    4. 以前のアーキテクチャの議論に接続
    
    私のユーザーID: {USER_ID}
    
    焦点を当てる：
    - 以前に議論したこととの関連性
    - 私の興味に関連する機能
    - 実用的なアプリケーション
    - AWSサービスの統合"""
)

print(response)

## セッション間理解

ベクトルメモリの力: 新しいセッションを開始しても、エージェントはまだ覚えています！

In [ ]:
# 新しいエージェントを作成（新しいセッションをシミュレート）
new_session_agent = Agent(
    model=bedrock_model,
    tools=[s3_vector_memory, image_reader, file_read, video_reader_local, use_llm],
    system_prompt=MEMORY_SYSTEM_PROMPT
)

# 以前の会話について尋ねる
response = new_session_agent(
    f"""私の興味について何を知っていますか？以前に何を議論しましたか？
    
    ユーザーID: {USER_ID}"""
)

print("🆕 新しいセッションの応答:")
print(response)

## メモリ操作

メモリ操作を詳しく見てみましょう：

In [ ]:
# ユーザーのすべてのメモリを一覧表示
result = s3_vector_memory(
    action="list",
    user_id=USER_ID
)

print(f"📊 保存されたメモリの合計: {result['total_found']}")
print("\nメモリ:")
for i, mem in enumerate(result.get('memories', []), 1):
    content = mem.get('memory', '')
    # 長いコンテンツを切り詰める
    display_content = content[:100] + '...' if len(content) > 100 else content
    print(f"\n{i}. {display_content}")
    print(f"   タイムスタンプ: {mem.get('created_at', 'N/A')}")

In [ ]:
# メモリ間のセマンティック検索
result = s3_vector_memory(
    action="retrieve",
    query="サーバーレスとAI技術",
    user_id=USER_ID,
    top_k=3
)

print("🔍 セマンティック検索結果:")
for i, mem in enumerate(result.get('memories', []), 1):
    print(f"\n{i}. {mem.get('memory', '')}")
    print(f"   関連性スコア: {mem.get('similarity', 'N/A')}")

## 仕組み

### アーキテクチャ概要

1. **インフラストラクチャセットアップ（自動）**
   - S3 Vectorバケットの存在を確認し、必要に応じて作成
   - ベクトルインデックスの存在を確認し、必要に応じて作成
   - 1024次元でインデックスを設定
   - 類似性にCOSINE距離メトリックを使用

2. **埋め込み生成**
   - テキストはAmazon Bedrock埋め込みを使用してベクトルに変換
   - モデル: `amazon.nova-2-multimodal-embeddings-v1:0`（最先端）
   - 1024次元のベクトルを生成
   - マルチモーダル埋め込みをサポート（テキスト、画像、動画、音声）

3. **ストレージ**
   - ベクトルはS3 Vectorsを使用してAmazon S3に保存
   - メタデータにはuser_id、タイムスタンプ、コンテンツが含まれる
   - 追跡のためのメモリごとの一意キー

4. **取得**
   - ベクトル類似性を使用したセマンティック検索
   - 最も関連性の高いメモリを返す
   - 分離のためにuser_idでフィルタリング

5. **分離**
   - user_idによるユーザー固有のメモリ
   - セキュアでスケーラブル
   - マルチテナント対応

## 使用例

### カスタマーサポート
- セッションをまたいで顧客の問題を記憶
- 一貫したサポートを提供
- 解決履歴を追跡

### パーソナルアシスタント
- 時間をかけてユーザーの好みを学習
- パーソナライズされた推奨事項を提供
- 日/週をまたいでコンテキストを維持

### コンテンツ分析
- 分析したコンテンツからの洞察を記憶
- ドキュメント間で情報を接続
- 時間をかけて知識を構築

### 開発ツール
- プロジェクトコンテキストを記憶
- 決定と根拠を追跡
- 一貫したガイダンスを提供

## クリーンアップ（オプション）

継続的な料金を避けるため、このノートブックで作成したS3 Vectorインデックスとバケットを削除できます。

In [ ]:
import boto3

# S3 Vectorsクライアントの初期化
s3vectors_client = boto3.client('s3vectors', region_name=AWS_REGION)

# ベクトルインデックスの削除
try:
    print(f"インデックスを削除中: {VECTOR_INDEX_NAME}...")
    s3vectors_client.delete_index(
        VectorBucketName=VECTOR_BUCKET_NAME,
        IndexName=VECTOR_INDEX_NAME
    )
    print(f"✅ インデックス '{VECTOR_INDEX_NAME}' を正常に削除しました")
except Exception as e:
    print(f"❌ インデックス削除エラー: {e}")

In [ ]:
# ベクトルバケットの削除
try:
    print(f"バケットを削除中: {VECTOR_BUCKET_NAME}...")
    s3vectors_client.delete_vector_bucket(
        VectorBucketName=VECTOR_BUCKET_NAME
    )
    print(f"✅ バケット '{VECTOR_BUCKET_NAME}' を正常に削除しました")
except Exception as e:
    print(f"❌ バケット削除エラー: {e}")

print("\n⚠️  注意: 削除が完了するまで数分かかる場合があります。")

## まとめ

このノートブックでは、以下を学習しました：

✅ 組み込みセッションマネージャーの制限

✅ Amazon S3 Vectorsがセッション間理解を可能にする方法

✅ s3_vector_memoryツールの設定と使用方法

✅ メモリ強化エージェントの作成方法

✅ メモリ間のセマンティック検索

✅ 本番環境対応のメモリアーキテクチャ

✅ 実世界の使用例

### 重要なポイント

1. **ベクトルメモリは真のAIパーソナライゼーションを可能にする**
2. **S3 Vectorsは本番環境対応のインフラストラクチャを提供**
3. **セマンティック検索はキーワードマッチングよりも強力**
4. **セッション間理解はより良いユーザー体験を創出**

### 次のステップ

- アプリケーションにベクトルメモリを実装
- 異なる埋め込みモデルを実験
- パーソナライズされたAI体験を構築
- 自信を持って本番環境にスケール！